In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import linprog
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("=== 问题1求解：单季种植优化模型 ===")

# 读取预处理完成的数据
stats_2023 = pd.read_csv('预处理完成的统计数据.csv', encoding='utf-8-sig')
compatibility_matrix = pd.read_csv('耕地_作物_兼容性_矩阵.csv', index_col=0, encoding='utf-8-sig')
land_df = pd.read_excel('D:/pythonproject/data_science/excel_files/附件1.xlsx', sheet_name='乡村的现有耕地')

# 获取基础数据
land_types = land_df['地块类型'].unique()
crop_ids = compatibility_matrix.index.astype(int)

# 计算各类耕地的总面积
land_area_by_type = land_df.groupby('地块类型')['地块面积/亩'].sum()

print("耕地类型及面积:")
for land_type in land_types:
    print(f"  {land_type}: {land_area_by_type[land_type]:.1f} 亩")

# 创建收益矩阵（作物编号 × 耕地类型）
profit_matrix = pd.DataFrame(index=crop_ids, columns=land_types)

# 填充收益矩阵（使用平均预期收益）
for crop_id in crop_ids:
    for land_type in land_types:
        # 查找该组合的预期收益
        matching_rows = stats_2023[
            (stats_2023['作物编号'] == crop_id) & 
            (stats_2023['地块类型'] == land_type) & 
            (stats_2023['种植季次'] == '单季')
        ]
        
        if len(matching_rows) > 0:
            profit = matching_rows['预期收益'].iloc[0]
        else:
            # 如果没有直接匹配的数据，使用该作物在其他地类的平均收益
            crop_profits = stats_2023[stats_2023['作物编号'] == crop_id]['预期收益']
            profit = crop_profits.mean() if len(crop_profits) > 0 else 0
        
        profit_matrix.loc[crop_id, land_type] = profit

print("\n收益矩阵创建完成:")
print(profit_matrix.head())

# 将兼容性矩阵和收益矩阵转换为数值形式
compat_matrix = compatibility_matrix.astype(float)
profit_matrix = profit_matrix.astype(float)

# 创建线性规划模型
print("\n=== 建立线性规划模型 ===")

# 变量数：每个作物在每种耕地上的种植面积
n_vars = len(crop_ids) * len(land_types)
print(f"变量数量: {n_vars}")

# 目标函数：最大化总收益
# c = -收益矩阵（因为 linprog 默认最小化）
c = []
for crop_id in crop_ids:
    for land_type in land_types:
        profit = profit_matrix.loc[crop_id, land_type]
        compat = compat_matrix.loc[crop_id, land_type]
        # 如果不兼容，则设置为很小的负值
        if compat == 0:
            c.append(1e6)  # 使得不兼容的组合不会被选中
        else:
            c.append(-profit)  # 最小化负收益相当于最大化收益

# 约束条件
# 1. 耕地面积约束：每种耕地的种植面积不能超过总面积
A_ub_land = []
b_ub_land = []

for i, land_type in enumerate(land_types):
    constraint = [0] * n_vars
    for j, crop_id in enumerate(crop_ids):
        idx = j * len(land_types) + i
        constraint[idx] = 1
    A_ub_land.append(constraint)
    b_ub_land.append(land_area_by_type[land_type])

# 2. 非负约束
bounds = [(0, None)] * n_vars

# 组合所有约束
A_ub = A_ub_land
b_ub = b_ub_land

print(f"约束条件数量: {len(A_ub)}")
print(f"变量上下界数量: {len(bounds)}")

# 求解线性规划问题
print("\n正在求解线性规划问题...")
result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

if result.success:
    print("优化成功！")
    print(f"最大总收益: {-result.fun:.2f} 元")
    
    # 解析优化结果
    solution = result.x
    
    # 将解析转换为可读形式
    planting_plan = pd.DataFrame(index=crop_ids, columns=land_types)
    
    for i, crop_id in enumerate(crop_ids):
        for j, land_type in enumerate(land_types):
            idx = i * len(land_types) + j
            area = solution[idx]
            if area > 1e-6:  # 只显示有意义的种植面积
                planting_plan.loc[crop_id, land_type] = area
            else:
                planting_plan.loc[crop_id, land_type] = 0
    
    # 获取作物名称
    crop_names = pd.read_excel('D:/pythonproject/data_science/excel_files/附件1.xlsx', 
                              sheet_name='乡村种植的农作物')[['作物编号', '作物名称']]
    planting_plan = planting_plan.merge(crop_names, left_index=True, right_on='作物编号')
    planting_plan.set_index(['作物编号', '作物名称'], inplace=True)
    
    print("\n=== 最优种植方案 ===")
    # 只显示有种植面积的组合
    non_zero_plan = planting_plan[(planting_plan > 1e-6).any(axis=1)]
    print(non_zero_plan.round(2))
    
    # 计算各类耕地的利用率
    utilization = {}
    for land_type in land_types:
        total_area = land_area_by_type[land_type]
        used_area = non_zero_plan[land_type].sum()
        utilization[land_type] = used_area / total_area * 100
    
    print("\n=== 耕地利用率 ===")
    for land_type, rate in utilization.items():
        print(f"{land_type}: {rate:.1f}%")
    
    # 保存种植方案
    non_zero_plan.to_csv('最优单季种植方案.csv', encoding='utf-8-sig')
    
else:
    print("优化失败:", result.message)

=== 问题1求解：单季种植优化模型 ===
耕地类型及面积:
  平旱地: 365.0 亩
  梯田: 619.0 亩
  山坡地: 108.0 亩
  水浇地: 109.0 亩
  普通大棚 : 9.6 亩
  智慧大棚: 2.4 亩



收益矩阵创建完成:
       平旱地       梯田      山坡地          水浇地        普通大棚          智慧大棚
1    900.0    835.0    770.0        835.0        835.0        835.0
2   3350.0   3162.5   2975.0       3162.5       3162.5       3162.5
3   2950.0   2785.0   2620.0       2785.0       2785.0       2785.0
4   2100.0   1960.0   1855.0  1971.666667  1971.666667  1971.666667
5  2451.25  2316.25  2181.25      2316.25      2316.25      2316.25

=== 建立线性规划模型 ===
变量数量: 246
约束条件数量: 6
变量上下界数量: 246

正在求解线性规划问题...
优化成功！
最大总收益: 3472675.00 元

=== 最优种植方案 ===
          平旱地     梯田    山坡地 水浇地 普通大棚  智慧大棚
作物编号 作物名称                                 
13   红薯     0  619.0  108.0   0     0    0

=== 耕地利用率 ===
平旱地: 0.0%
梯田: 100.0%
山坡地: 100.0%
水浇地: 0.0%
普通大棚 : 0.0%
智慧大棚: 0.0%
